In [ ]:
#1) Imports & GPU Check
import os, time, tarfile, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from IPython.display import clear_output

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.utils as vutils

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'   GPU : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print(' No GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
#2) Download & Preview Facades Dataset
# ── Config ────────────────────────────────────
# Change DATASET_NAME to try other datasets:
#   'maps'        → satellite photo  ↔  map
#   'edges2shoes' → edge sketch      ↔  shoe photo
#   'cityscapes'  → segmentation map ↔  street scene
DATASET_NAME = 'facades'
DATA_ROOT    = '/content/datasets'
DATASET_URL  = f'http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/{DATASET_NAME}.tar.gz'

os.makedirs(DATA_ROOT, exist_ok=True)
tar_path = f'{DATA_ROOT}/{DATASET_NAME}.tar.gz'
DATA_DIR = f'{DATA_ROOT}/{DATASET_NAME}'

if not Path(DATA_DIR, 'train').exists():
    print(f' Downloading {DATASET_NAME} (~30 MB)…')
    urllib.request.urlretrieve(DATASET_URL, tar_path)
    print('Extracting…')
    with tarfile.open(tar_path) as t:
        t.extractall(DATA_ROOT)
    os.remove(tar_path)
    print('Done!')
else:
    print('Dataset already downloaded.')

# ── Preview 3 sample pairs ────────────────────────────────────────────────────
samples = sorted(Path(DATA_DIR, 'train').glob('*.jpg'))[:3]
fig, axes = plt.subplots(3, 2, figsize=(8, 9))
fig.suptitle('Left: Real Photo (Target)  |  Right: Label Map (Input)', fontsize=11)

for i, f in enumerate(samples):
    img = np.array(Image.open(f))
    w = img.shape[1] // 2
    axes[i,0].imshow(img[:, :w]); axes[i,0].axis('off'); axes[i,0].set_title('Real Photo (Target)')
    axes[i,1].imshow(img[:, w:]); axes[i,1].axis('off'); axes[i,1].set_title('Label Map (Input)')
plt.tight_layout(); plt.show()

n_train = len(list(Path(DATA_DIR,'train').glob('*.jpg')))
n_val   = len(list(Path(DATA_DIR,'val').glob('*.jpg')))
print(f'Train: {n_train} images  |  Val: {n_val} images')

In [ ]:
#3) Dataset Class & DataLoaders
IMG_SIZE   = 256
BATCH_SIZE = 4   # increase to 8 if VRAM > 8 GB

class Pix2PixDataset(Dataset):
    """
    Loads 512×256 side-by-side paired images and splits them into:
      - Left  256×256 → input  (segmentation map / edge map)
      - Right 256×256 → target (real photo)
    Training uses random jitter (resize to 286 → crop to 256) + random flip.
    """
    def __init__(self, root_dir, split='train', img_size=256, augment=True):
        self.files   = sorted(Path(root_dir, split).glob('*.jpg')) + \
                       sorted(Path(root_dir, split).glob('*.png'))
        self.size    = img_size
        self.augment = augment and (split == 'train')
        self.to_tensor = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),  # → [-1, 1]
        ])

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        img  = Image.open(self.files[idx]).convert('RGB')
        w, h = img.size
        half = w // 2
        tgt = img.crop((0,    0, half, h))
        inp = img.crop((half, 0, w,    h))

        if self.augment:
            jitter = self.size + 30
            tgt = tgt.resize((jitter, jitter), Image.BICUBIC)
            inp = inp.resize((jitter, jitter), Image.BICUBIC)
            i = np.random.randint(0, jitter - self.size)
            j = np.random.randint(0, jitter - self.size)
            tgt = tgt.crop((j, i, j+self.size, i+self.size))
            inp = inp.crop((j, i, j+self.size, i+self.size))
            if np.random.rand() > 0.5:
                tgt = tgt.transpose(Image.FLIP_LEFT_RIGHT)
                inp = inp.transpose(Image.FLIP_LEFT_RIGHT)
        else:
            tgt = tgt.resize((self.size, self.size), Image.BICUBIC)
            inp = inp.resize((self.size, self.size), Image.BICUBIC)

        return self.to_tensor(inp), self.to_tensor(tgt)


train_ds = Pix2PixDataset(DATA_DIR, 'train', IMG_SIZE, augment=True)
val_ds   = Pix2PixDataset(DATA_DIR, 'val',   IMG_SIZE, augment=False)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=1,          shuffle=False, num_workers=2)

print(f'Train batches: {len(train_dl)}  |  Val images: {len(val_ds)}')

In [ ]:
#4) Generator (U-Net) + Discriminator (PatchGAN)
# ══════════════════════════════════════════════════════════════════════════════
#  GENERATOR — U-Net with Skip Connections
#  Encoder: 8 downsampling blocks  (256×256 → 1×1 bottleneck)
#  Decoder: 8 upsampling blocks    (1×1 → 256×256)
#  Skip connections: each encoder feature map is concatenated to its
#  mirror decoder layer → preserves spatial structure (edges, layout)
# ══════════════════════════════════════════════════════════════════════════════

class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, down=True, use_bn=True, dropout=False, relu=True):
        super().__init__()
        layers = []
        if down:
            layers.append(nn.Conv2d(in_ch, out_ch, 4, stride=2, padding=1, bias=not use_bn))
        else:
            layers.append(nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1, bias=not use_bn))
        if use_bn:  layers.append(nn.BatchNorm2d(out_ch))
        if dropout: layers.append(nn.Dropout(0.5))
        layers.append(nn.ReLU(True) if relu else nn.LeakyReLU(0.2, True))
        self.block = nn.Sequential(*layers)

    def forward(self, x): return self.block(x)


class UNetGenerator(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, ngf=64):
        super().__init__()
        # Encoder (no BN on first layer)
        self.e1 = nn.Sequential(nn.Conv2d(in_ch, ngf, 4, 2, 1), nn.LeakyReLU(0.2, True))  # 128
        self.e2 = UNetBlock(ngf,   ngf*2, relu=False)   # 64
        self.e3 = UNetBlock(ngf*2, ngf*4, relu=False)   # 32
        self.e4 = UNetBlock(ngf*4, ngf*8, relu=False)   # 16
        self.e5 = UNetBlock(ngf*8, ngf*8, relu=False)   # 8
        self.e6 = UNetBlock(ngf*8, ngf*8, relu=False)   # 4
        self.e7 = UNetBlock(ngf*8, ngf*8, relu=False)   # 2
        self.e8 = nn.Sequential(nn.Conv2d(ngf*8, ngf*8, 4, 2, 1), nn.ReLU(True))    # 1 bottleneck
        # Decoder (in_ch doubled at each layer due to skip concat)
        self.d1 = UNetBlock(ngf*8,   ngf*8, down=False, dropout=True)   # 2
        self.d2 = UNetBlock(ngf*16,  ngf*8, down=False, dropout=True)   # 4
        self.d3 = UNetBlock(ngf*16,  ngf*8, down=False, dropout=True)   # 8
        self.d4 = UNetBlock(ngf*16,  ngf*8, down=False)                 # 16
        self.d5 = UNetBlock(ngf*16,  ngf*4, down=False)                 # 32
        self.d6 = UNetBlock(ngf*8,   ngf*2, down=False)                 # 64
        self.d7 = UNetBlock(ngf*4,   ngf,   down=False)                 # 128
        self.d8 = nn.Sequential(nn.ConvTranspose2d(ngf*2, out_ch, 4, 2, 1), nn.Tanh())    # 256

    def forward(self, x):
        e1=self.e1(x); e2=self.e2(e1); e3=self.e3(e2); e4=self.e4(e3)
        e5=self.e5(e4); e6=self.e6(e5); e7=self.e7(e6); e8=self.e8(e7)
        d1 = torch.cat([self.d1(e8), e7], 1)
        d2 = torch.cat([self.d2(d1), e6], 1)
        d3 = torch.cat([self.d3(d2), e5], 1)
        d4 = torch.cat([self.d4(d3), e4], 1)
        d5 = torch.cat([self.d5(d4), e3], 1)
        d6 = torch.cat([self.d6(d5), e2], 1)
        d7 = torch.cat([self.d7(d6), e1], 1)
        return self.d8(d7)


# ══════════════════════════════════════════════════════════════════════════════
#  DISCRIMINATOR — 70×70 PatchGAN
#  Receives (input_image + real_or_fake) concatenated → 6 channels
#  Output: 30×30 map — each value = real/fake score for that 70×70 patch
# ══════════════════════════════════════════════════════════════════════════════

class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_ch=3, ndf=64):
        super().__init__()
        def blk(ic, oc, s=2, bn=True):
            l = [nn.Conv2d(ic, oc, 4, s, 1, bias=not bn)]
            if bn: l.append(nn.BatchNorm2d(oc))
            l.append(nn.LeakyReLU(0.2, True))
            return l
        self.model = nn.Sequential(
            *blk(in_ch*2, ndf,   bn=False),   # (N,  64, 128, 128)
            *blk(ndf,   ndf*2),               # (N, 128,  64,  64)
            *blk(ndf*2, ndf*4),               # (N, 256,  32,  32)
            *blk(ndf*4, ndf*8, s=1),          # (N, 512,  31,  31)
            nn.Conv2d(ndf*8, 1, 4, 1, 1),     # (N,   1,  30,  30)
        )
    def forward(self, inp, tgt):
        return self.model(torch.cat([inp, tgt], 1))


# ── Weight init (normal distribution, std=0.02) ──────────────────────────────
def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None: nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

G = UNetGenerator().to(device);          G.apply(weights_init)
D = PatchGANDiscriminator().to(device);  D.apply(weights_init)

print(f'Generator params: {sum(p.numel() for p in G.parameters()):,}')
print(f'Discriminator params: {sum(p.numel() for p in D.parameters()):,}')

In [ ]:
#5) Hyperparameters, Loss & Optimizers
EPOCHS          = 100    # 200 for best quality; 100 gives solid results on T4
LR              = 2e-4
LAMBDA_L1       = 100    # how much to weight pixel accuracy vs realism
SAMPLE_INTERVAL = 5      # show progress image every N epochs
SAVE_INTERVAL   = 20     # save checkpoint every N epochs

CKPT_DIR   = '/content/checkpoints'
SAMPLE_DIR = '/content/samples'
os.makedirs(CKPT_DIR,   exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)

criterion_adv = nn.BCEWithLogitsLoss()   # adversarial: real vs fake patches
criterion_l1  = nn.L1Loss()             # pixel-level reconstruction fidelity

opt_G = optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))

print(f'Ready  |  Epochs={EPOCHS}  LR={LR}  λ_L1={LAMBDA_L1}  Batch={BATCH_SIZE}')

In [ ]:
#6) Training Loop
history = {'D': [], 'G': [], 'G_adv': [], 'G_l1': []}

# Fixed val sample for consistent visual tracking across epochs
fixed_inp, fixed_real = next(iter(val_dl))
fixed_inp  = fixed_inp.to(device)
fixed_real = fixed_real.to(device)

def denorm(t):
    """Convert [-1,1] tensor → displayable numpy image."""
    return (t.squeeze(0).cpu().permute(1,2,0).numpy() * 0.5 + 0.5).clip(0, 1)

print('Training started…\n')
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    G.train(); D.train()
    d_ep, g_ep, adv_ep, l1_ep = [], [], [], []

    for inp, real in train_dl:
        inp  = inp.to(device)
        real = real.to(device)

        # ── Discriminator update ─────────────────────────────────────────
        with torch.no_grad():
            fake = G(inp)

        pred_real = D(inp, real)
        pred_fake = D(inp, fake.detach())

        # dynamic label shape — works regardless of img size
        real_lbl = torch.ones_like(pred_real)
        fake_lbl = torch.zeros_like(pred_fake)

        # Reuse pred_real directly instead of recomputing
        loss_D = 0.5 * (criterion_adv(pred_real, real_lbl) +
                         criterion_adv(pred_fake, fake_lbl))
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()

        # ── Generator update ─────────────────────────────────────────────
        fake    = G(inp)
        pred_fake_for_G = D(inp, fake)
        real_lbl_G = torch.ones_like(pred_fake_for_G)

        g_adv  = criterion_adv(pred_fake_for_G, real_lbl_G)
        g_l1   = criterion_l1(fake, real) * LAMBDA_L1
        loss_G = g_adv + g_l1
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()

        d_ep.append(loss_D.item());  g_ep.append(loss_G.item())
        adv_ep.append(g_adv.item()); l1_ep.append(g_l1.item())

    d_m, g_m = np.mean(d_ep), np.mean(g_ep)
    history['D'].append(d_m);      history['G'].append(g_m)
    history['G_adv'].append(np.mean(adv_ep))
    history['G_l1'].append(np.mean(l1_ep))

    elapsed = (time.time() - t0) / 60
    print(f'Epoch [{epoch:3d}/{EPOCHS}]  D={d_m:.4f}  G={g_m:.4f}  '
          f'(adv {np.mean(adv_ep):.3f} | l1 {np.mean(l1_ep):.2f})  [{elapsed:.1f} min]')

    # ── Inline visualisation ─────────────────────────────────────────────
    if epoch % SAMPLE_INTERVAL == 0 or epoch == 1:
        G.eval()
        with torch.no_grad():
            fake_val = G(fixed_inp)

        clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(13, 4))
        fig.suptitle(f'Epoch {epoch}/{EPOCHS}   D_loss={d_m:.4f}   G_loss={g_m:.4f}', fontsize=12)
        for ax, img, title in zip(axes,
                                   [fixed_inp, fake_val, fixed_real],
                                   ['Input (label map)', ' Generated (fake)', 'Target (real)']):
            ax.imshow(denorm(img)); ax.set_title(title); ax.axis('off')
        plt.tight_layout()
        plt.savefig(f'{SAMPLE_DIR}/epoch_{epoch:04d}.png', dpi=100)
        plt.show()

    # ── Save checkpoints ─────────────────────────────────────────────────
    if epoch % SAVE_INTERVAL == 0:
        torch.save(G.state_dict(), f'{CKPT_DIR}/G_epoch{epoch}.pth')
        torch.save(D.state_dict(), f'{CKPT_DIR}/D_epoch{epoch}.pth')
        print(f'   Saved checkpoints at epoch {epoch}')

print(f'\n Done! Total time: {(time.time()-t0)/60:.1f} min')

In [ ]:
#7) Plot Loss Curves
epochs_x = range(1, len(history['D']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(epochs_x, history['D'], label='Discriminator', color='crimson')
ax1.plot(epochs_x, history['G'], label='Generator (total)', color='royalblue')
ax1.set_title('Training Losses'); ax1.set_xlabel('Epoch')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_x, history['G_adv'], label='G adversarial', color='orange')
ax2.plot(epochs_x, history['G_l1'],  label=f'G L1 (×{LAMBDA_L1})', color='seagreen')
ax2.set_title('Generator Loss Components'); ax2.set_xlabel('Epoch')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/loss_curves.png', dpi=120)
plt.show()

In [ ]:
#8) Inference on Validation Set
RESULTS_DIR = '/content/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

G.eval()
with torch.no_grad():
    for i, (inp, real) in enumerate(val_dl):
        inp  = inp.to(device)
        fake = G(inp)
        grid = vutils.make_grid(
            torch.cat([inp.cpu(), fake.cpu(), real], dim=0),
            nrow=3, normalize=True, value_range=(-1, 1)
        )
        vutils.save_image(grid, f'{RESULTS_DIR}/result_{i:04d}.png')

print(f'Saved {len(val_ds)} results to {RESULTS_DIR}/')

# Show a 3×3 grid
result_files = sorted(Path(RESULTS_DIR).glob('*.png'))[:9]
fig, axes = plt.subplots(3, 3, figsize=(13, 13))
fig.suptitle('Columns: Input | Generated | Real', fontsize=12)
for ax, f in zip(axes.flat, result_files):
    ax.imshow(np.array(Image.open(f))); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
#9) Download Results
from google.colab import files

!zip -r /content/pix2pix_checkpoints.zip /content/checkpoints
files.download('/content/pix2pix_checkpoints.zip')

!zip -r /content/pix2pix_results.zip /content/results /content/samples /content/loss_curves.png
files.download('/content/pix2pix_results.zip')